# [심화·선택] 06. Agents (Advanced) — 제품 탑재형 AI 고객지원/원격진단 시나리오
**ToolRuntime(Context) + 권한(Authorization) + Structured Output + 멀티턴 메모리**

---

## 🧩 시나리오(가상)
스마트 생활가전/로보틱스 제품에는 **고객지원(FAQ/매뉴얼) + 원격진단 + 서비스 티켓 생성**이 필요합니다.

이 노트북은 에이전트가 아래를 수행하도록 만듭니다.

- 고객 문의에서 **기기 ID/증상/에러코드**를 추출
- 필요하면 **매뉴얼/정책 도구**로 조회
- 권한이 있을 때만 **원격진단/티켓 생성/펌웨어 푸시** 같은 액션 실행
- 결과는 **Structured Output(JSON)** 로 반환(채팅 UI/앱에 안전하게 탑재 가능)

> 모든 데이터는 더미이며, 실제 기업 데이터/플로우와 무관합니다.

> **사전 조건 (로컬 LLM)**: `03_llama_server_local_llm.ipynb`에서 띄운 **llama-server(http://localhost:8080 · 기본 Qwen3.5-4B / 옵션 Qwen2.5-7B)** 가 실행 중이어야 합니다.
> 서버가 꺼져 있다면 터미널에서 `python local_server.py start 4b` 를 실행하세요. (API 키/비용 없음, 모든 추론은 내 GPU에서 수행)



이 노트북은 6시간 핵심 과정 이후 진행하는 심화 선택 자료입니다.


이 노트북은 6시간 핵심 과정 이후 진행하는 심화 선택 자료입니다.


## 🧭 시작 전에

| 항목 | 내용 |
|---|---|
| 🎯 학습 목표 | LLM이 스스로 도구를 골라 실행하는 **에이전트**를 만들고, 권한 제어·구조화 응답·대화 메모리를 붙인다 |
| 📚 사전 지식 | 03번 7~8절(도구 호출), 05번(구조화 출력) |
| 🐍 커널(환경) | `LLM Hands-On (Win CUDA)` |
| 🖥️ GPU 사용 | 공용 llama-server |
| ⏱️ 소요 시간 | 약 50분 |
| ✔️ 실행 전 확인 | llama-server 실행 중 |

> **실행 요령**: 위에서부터 **한 셀씩**(Shift+Enter) 실행하세요. `Run All`은 쓰지 마세요.
> `✅ TODO 셀`은 학생이 채우는 칸이라 그대로 실행하면 일부러 에러가 납니다. 막히면 바로 아래 `✅ 참고 답안` 셀을 보세요.

## 📖 핵심 개념: 에이전트 = LLM + 도구 + 반복

```
          ┌──────────────────────────────────────────────┐
 질문 ──▶ │ LLM: "get_device_info(DEV-2002)를 불러야겠다"  │
          └───────────────┬──────────────────────────────┘
                          ▼ (tool call)
                  우리 코드가 함수 실행 → 결과
                          │
          ┌───────────────▼──────────────────────────────┐
          │ LLM: 결과를 보고 다음 행동 결정               │ ── 도구가 더 필요하면 반복
          └───────────────┬──────────────────────────────┘
                          ▼ (더 필요 없으면)
              최종 답변 (SupportResponse JSON)
```

| 이 노트북의 부품 | 역할 |
|---|---|
| `@tool` 함수 | LLM이 쓸 수 있는 기능. **함수 이름·docstring·타입 힌트가 그대로 LLM에게 설명서로 전달**됨 |
| `ToolRuntime[SupportContext]` | 사용자 역할(role) 같은 **실행 맥락**을 도구에 몰래 전달. LLM은 이 값을 보거나 조작할 수 없음 → 권한 검사에 안전 |
| `response_format=ToolStrategy(...)` | 05번에서 배운 구조화 최종 응답 |
| `checkpointer` + `thread_id` | 대화 기록 저장. 같은 thread_id면 이전 대화를 기억 (03번: 모델 자체는 기억 못 함) |
| `recursion_limit` | 도구 호출을 무한 반복하지 않도록 하는 안전장치 |

> **왜 권한을 프롬프트가 아니라 도구 코드에서 검사할까?** 프롬프트 규칙은 모델이 어길 수 있지만, 코드의 `if role == "customer": 거부`는 절대 어길 수 없기 때문입니다. 위험한 작업은 항상 코드로 막습니다.

In [1]:
# !pip -q install -U langchain langchain-openai langgraph pydantic requests


In [2]:
import os, getpass
from dataclasses import dataclass
from typing import Optional, Literal, List

from langchain.tools import tool, ToolRuntime
# tool은 데코레이터
# @tool
# def 함수_이름(...)
#    함수 로직...
#    return
# 그리고 나중에 create_agent 호출할 때 tools 인자에 함수 리스트 넘기기
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
# 출력 형식 강제하고자 할 때 create_agent의 response_format에 ToolStrategy 인스턴스 넘기기

from langgraph.checkpoint.memory import InMemorySaver
# 대화 히스토리를 기억할 채로 채팅하게 만들고 싶을 때 create_agent의 checkpointer 인자에 InMemorySaver 인스턴스 넘기기

from pydantic import BaseModel, Field
from langchain_core.messages import AIMessage
from langgraph.errors import GraphRecursionError

# ── 로컬 LLM 연결 (공용 설정: local_llm.py) ──
# 모든 앱 실습(05~09)은 같은 로컬 OpenAI 호환 서버(llama-server)를 사용합니다.
from local_llm import get_chat_model, check_server, LLM_BASE_URL, LLM_MODEL
check_server()   # 서버가 안 떠 있으면 해결 방법을 알려주고 에러를 냅니다.

# 로컬 LLM (llama-server, 4B/7B 프리셋). 에이전트는 tool 선택이 흔들리지 않도록 temperature=0
model = get_chat_model(temperature=0)


✅ 로컬 LLM 서버 연결 OK: http://localhost:8080/v1
   서버에 로드된 모델: Qwen3.5-4B-Q8_0.gguf


## 1) Runtime Context & 권한 모델

- 고객/상담사/관리자 등 **role**에 따라 가능한 액션이 달라집니다.
- 예: `run_remote_diagnosis`, `create_service_ticket` 같은 원격 지원 액션은 `role="support_agent"` 이상만 가능하도록 제한

또한 locale 같은 맥락도 컨텍스트로 주입할 수 있습니다.


In [3]:
class AuthorizationError(Exception):
    pass

@dataclass
class SupportContext:
    user_id: str
    user_name: str
    role: Literal["customer", "support_agent", "admin"]
    locale: Literal["ko", "en"] = "ko"

# 더미 기기/보증 DB (실데이터 아님)
DEV_DB = {
    "DEV-1001": {"model": "RobotVacuum-X", "status": "online", "fw": "1.2.0", "warranty_days_left": 120},
    "DEV-2002": {"model": "Washer-A", "status": "online", "fw": "2.1.7", "warranty_days_left": 0, "last_error": "5C"},
    "DEV-3003": {"model": "AirCon-Pro", "status": "online", "fw": "3.5.1", "warranty_days_left": 420, "filter_warning": True},
}

POLICY = {
    "warranty": "무상 보증은 제품/부품에 따라 기간이 다를 수 있으며, 외관 손상/소모품은 제외될 수 있습니다. (강의용 더미 정책)",
    "service_visit": "기사 방문 예약은 지역/일정에 따라 달라질 수 있습니다. 개인정보는 채팅에 남기지 않고 안전한 채널로 안내합니다.",
}

TICKETS = []


## 2) Tools 구현 (권한/컨텍스트 활용)

제품 탑재형/제품 지원 시나리오에서는 LLM이 직접 '실행'하지 않고,
**명확히 정의된 도구(API/SDK)** 를 호출하여 기기 상태를 조회하거나 원격진단을 수행합니다.

여기서는 네트워크/실기기 없이도 실습할 수 있도록, 아래 도구를 **더미 DB** 기반으로 구현합니다.

- `get_device_info(device_id)` : 상태/펌웨어/보증/최근 에러코드 조회
- `lookup_policy(topic)` : 보증/기사방문 안내 문구 조회
- `run_remote_diagnosis(device_id)` : (권한 필요) 원격진단
- `create_service_ticket(device_id, summary)` : (권한 필요) 티켓 생성

- `push_firmware_update(device_id, target_version, confirmed)` : (관리자 전용/위험) 펌웨어 업데이트 푸시


In [4]:
@tool
def get_device_info(device_id: str) -> str:
    """기기 ID로 상태/펌웨어/보증 등을 조회합니다.
    - device_id 예: DEV-2002
    """
    dev = DEV_DB.get(device_id)
    if not dev:
        return f"NOT_FOUND: {device_id}"
    return (
        f"{device_id} | model={dev.get('model')} | status={dev.get('status')} | "
        f"fw={dev.get('fw')} | warranty_days_left={dev.get('warranty_days_left')} | "
        f"last_error={dev.get('last_error')} | filter_warning={dev.get('filter_warning')}"
    )


@tool
def lookup_policy(topic: Literal["warranty","service_visit"]) -> str:
    """정책/안내 문구(더미)를 조회합니다."""
    return POLICY.get(topic, "NOT_FOUND")


@tool
def run_remote_diagnosis(runtime: ToolRuntime[SupportContext], device_id: str) -> str:
    """(상담사 권한 필요) 원격 자가진단을 실행합니다."""
    if runtime.context.role == "customer":
        return "AUTH_ERROR: customer는 원격진단 권한이 없습니다. 사람 상담사 연결이 필요합니다."
    dev = DEV_DB.get(device_id)
    if not dev:
        return f"NOT_FOUND: {device_id}"
    if dev.get("last_error") == "5C":
        return "DIAG: drain_issue_likely | action: check_filter_and_hose"
    if dev.get("filter_warning"):
        return "DIAG: filter_warning=true | action: clean_filter"
    return "DIAG: no_critical_issue"


@tool
def create_service_ticket(runtime: ToolRuntime[SupportContext], device_id: str, summary: str) -> str:
    """(상담사 권한 필요) 서비스 티켓을 생성합니다. 개인정보는 포함하지 마세요."""
    if runtime.context.role == "customer":
        return "AUTH_ERROR: customer는 티켓 생성 권한이 없습니다. 사람 상담사 연결이 필요합니다."
    ticket = {"ticket_id": f"TCK-{1000+len(TICKETS)}", "device_id": device_id, "summary": summary[:200]}
    TICKETS.append(ticket)
    return f"OK: {ticket}"


@tool
def push_firmware_update(runtime: ToolRuntime[SupportContext], device_id: str, target_version: str, confirmed: bool = False) -> str:
    """(관리자 전용/위험) 펌웨어 업데이트를 푸시합니다. confirmed=true일 때만 진행합니다."""
    if runtime.context.role != "admin":
        return "AUTH_ERROR: admin만 펌웨어 업데이트 권한이 있습니다."
    if not confirmed:
        return "CONFIRMATION_REQUIRED: 펌웨어 업데이트는 위험 작업입니다. confirmed=true로 다시 호출해야 합니다."
    dev = DEV_DB.get(device_id)
    if not dev:
        return f"NOT_FOUND: {device_id}"
    dev["fw"] = target_version
    return f"OK: updated_fw={target_version}"


### 🔍 들여다보기: LLM에게 도구가 어떻게 보일까?

아래는 LLM에게 실제로 전달되는 도구 설명서입니다. `runtime` 인자는 **보이지 않는다**는 점에 주목하세요 (LLM이 역할을 위조할 수 없음).

In [5]:
import json
for t in [get_device_info, run_remote_diagnosis, push_firmware_update]:
    print(f"■ {t.name}: {t.description}")
    print(json.dumps(t.tool_call_schema.model_json_schema().get('properties', {}), ensure_ascii=False, indent=2))

■ get_device_info: 기기 ID로 상태/펌웨어/보증 등을 조회합니다.
- device_id 예: DEV-2002
{
  "device_id": {
    "title": "Device Id",
    "type": "string"
  }
}
■ run_remote_diagnosis: (상담사 권한 필요) 원격 자가진단을 실행합니다.
{
  "device_id": {
    "title": "Device Id",
    "type": "string"
  }
}
■ push_firmware_update: (관리자 전용/위험) 펌웨어 업데이트를 푸시합니다. confirmed=true일 때만 진행합니다.
{
  "device_id": {
    "title": "Device Id",
    "type": "string"
  },
  "target_version": {
    "title": "Target Version",
    "type": "string"
  },
  "confirmed": {
    "default": false,
    "title": "Confirmed",
    "type": "boolean"
  }
}


## 3) 최종 구조화 응답 스키마

프론트/UI에서 바로 사용할 수 있도록, 최종 응답을 구조화합니다.


In [6]:
class ActionItem(BaseModel):
    action: Literal[
        "asked_question",
        "looked_up_policy",
        "checked_device",
        "ran_diagnosis",
        "created_ticket",
        "none"
    ]
    detail: str

class SupportResponse(BaseModel):
    reply: str = Field(description="사용자에게 보여줄 답변(존댓말)")
    needs_human: bool = Field(description="사람 상담사/기사 연결이 필요한가")
    action_items: List[ActionItem] = Field(default_factory=list, description="이번 턴에 수행/제안한 액션 기록")


## 4) Agent 구성 (System Prompt 강화 + Memory)

- **Structured Output**: 응답을 `SupportResponse(JSON)`로 강제하면 앱/제품 UI에 안전하게 탑재하기 쉽습니다.
- **Memory**: 같은 `thread_id`로 여러 턴을 이어가면, 사용자가 기기 ID를 생략해도(“아까 그 세탁기요”) 맥락을 유지할 수 있습니다.
- **권한 가드**: 고객(role=customer)이 위험/권한 필요한 요청을 하면, 실행하지 않고 사람 연결을 유도합니다.


In [7]:
SYSTEM_PROMPT = """당신은 '스마트 생활가전/로보틱스' 제품 고객지원 에이전트입니다.
- tool 결과가 AUTH_ERROR로 시작하면 권한 없는 작업이므로 실행을 중단하고, needs_human=true로 설정한 뒤 사람 상담사 연결을 안내하세요.
규칙:
1) 사용자가 보증/기사방문 등 정책을 물으면 lookup_policy 도구를 사용하세요.
2) 기기 상태/보증 확인이 필요하면 get_device_info를 사용하세요. 기기 ID(DEV-xxxx)가 없으면 먼저 질문하세요.
3) run_remote_diagnosis / create_service_ticket는 상담사(role=support_agent/admin)만 호출 가능합니다.
   - 고객(role=customer)이라면 needs_human=true로 설정하고 사람 연결을 유도하세요.
4) push_firmware_update는 관리자(role=admin)만 가능하며, confirmed=true가 없으면 반드시 확인 질문을 먼저 하세요.
5) 개인정보(주소/연락처/시리얼 등)는 채팅에 재노출하지 말고 안전한 채널로 안내하세요.
6) 최종 응답은 SupportResponse 스키마(JSON)로만 출력하세요.
"""

checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    tools=[get_device_info, lookup_policy, run_remote_diagnosis, create_service_ticket, push_firmware_update],
    system_prompt=SYSTEM_PROMPT,
    context_schema=SupportContext,
    response_format=ToolStrategy(SupportResponse),
    checkpointer=checkpointer,
)

def invoke_agent(user_text: str, *, thread_id: str, ctx: SupportContext):
    # recursion_limit: 로컬 모델이 tool 호출을 반복하는 경우를 대비한 안전 상한
    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": 25}
    try:
        res = agent.invoke(
            {"messages": [{"role": "user", "content": user_text}]},
            config=config,
            context=ctx,
        )
    except GraphRecursionError:
        # 한도 초과 시 셀 전체가 멈추지 않도록, 지금까지의 대화 상태를 돌려줌 (show()가 처리)
        state = agent.get_state(config).values
        res = {"messages": state.get("messages", []) or [AIMessage(content="(재귀 한도 초과)")],
               "structured_response": None}
        print("⚠ recursion_limit 초과 — 로컬 모델이 tool 호출을 반복했습니다. (프롬프트 개선 포인트)")
    return res

def show(res):
    """구조화 응답 출력. (로컬 모델이 스키마 tool 대신 일반 텍스트로 끝낸 경우에도 내용을 보여줌)"""
    sr = res.get("structured_response")
    print(sr if sr is not None else f"(structured_response 없음) {res['messages'][-1].content}")


## 5) 실행 예시 (권한에 따른 차이 관찰)

- customer role로는 원격진단/티켓 생성 같은 액션을 못 하므로 `needs_human=true`로 연결 안내가 나와야 합니다.
- support_agent role로는 원격진단/티켓 생성이 가능해야 합니다.


In [8]:
# 1) 고객(권한 없음) — 원격진단/티켓 생성 불가 → 사람 연결 유도
ctx_customer = SupportContext(user_id="u-1", user_name="민수", role="customer", locale="ko")
r1 = invoke_agent("DEV-2002 세탁기 5C 에러가 떠요. 원격진단도 해주시고, 필요하면 티켓도 만들어주세요.", thread_id="demo-1", ctx=ctx_customer)
show(r1)

# 2) 상담사(권한 있음) — 원격진단 + 티켓 생성 가능
ctx_agent = SupportContext(user_id="agent-7", user_name="지원", role="support_agent", locale="ko")
r2 = invoke_agent("DEV-2002 원격진단 실행하고, 결과 기반으로 티켓 생성까지 해주세요.", thread_id="demo-2", ctx=ctx_agent)
show(r2)

# 3) 관리자(권한 있음) — 위험 작업(펌웨어 업데이트)은 확인 질문이 먼저 나와야 함
ctx_admin = SupportContext(user_id="admin-1", user_name="관리자", role="admin", locale="ko")
r3 = invoke_agent("DEV-3003 에어컨 펌웨어를 3.5.2로 올려줘.", thread_id="demo-3", ctx=ctx_admin)
show(r3)


reply='안녕하세요. 원격진단 권한이 고객님에게는 제공되지 않으므로, 사람 상담사(기사)와 연결하여 직접 진행해 드릴 수 있습니다. 잠시만 기다려 주세요.' needs_human=True action_items=[ActionItem(action='none', detail='원격진단 권한 확인 및 사람 상담사 연결 필요성 판단')]
reply='안녕하세요! 원격진단을 완료하고 결과를 확인했습니다. 진단 결과 배수 문제 가능성이 높으며, 필터와 호스를 점검하는 것이 필요하다고 나타났습니다. 이 결과를 바탕으로 서비스 티켓(TCK-1000)을 생성하여 관련 전문가가 점검을 진행하도록 안내했습니다. 추가적으로 도움이 필요하시면 언제든지 말씀해 주세요!' needs_human=False action_items=[ActionItem(action='ran_diagnosis', detail='DEV-2002 원격진단 실행'), ActionItem(action='created_ticket', detail='원격진단 결과 기반 티켓 생성 (TCK-1000)')]
reply='안녕하세요. 펌웨어 업데이트는 위험 작업이므로, 반드시 확인 후 진행해야 합니다. 업데이트를 진행하시겠습니까?' needs_human=False action_items=[ActionItem(action='none', detail='펌웨어 업데이트 확인 요청')]


### 🔍 들여다보기: 에이전트가 실제로 한 일 (실행 기록)

결과만 보면 에이전트가 어떻게 생각했는지 알 수 없습니다. 메시지 기록을 순서대로 출력해 봅시다.
- `HumanMessage`: 사용자 입력 / `AIMessage`: LLM 출력 (도구 호출 요청 포함) / `ToolMessage`: 도구 실행 결과

In [9]:
def print_trace(res):
    for m in res["messages"]:
        kind = type(m).__name__
        calls = getattr(m, "tool_calls", None)
        if calls:
            print(f"[{kind}] 도구 호출 → " + ", ".join(f"{c['name']}({c['args']})" for c in calls))
        else:
            print(f"[{kind}] {str(m.content)[:200]}")

print("=== 상담사(support_agent) 요청의 실행 기록 ===")
print_trace(r2)

=== 상담사(support_agent) 요청의 실행 기록 ===
[HumanMessage] DEV-2002 원격진단 실행하고, 결과 기반으로 티켓 생성까지 해주세요.
[AIMessage] 도구 호출 → run_remote_diagnosis({'device_id': 'DEV-2002'})
[ToolMessage] DIAG: drain_issue_likely | action: check_filter_and_hose
[AIMessage] 도구 호출 → create_service_ticket({'device_id': 'DEV-2002', 'summary': '원격진단 결과: 배수 문제 가능성 높음 (필터/호스 점검 필요)'})
[ToolMessage] OK: {'ticket_id': 'TCK-1000', 'device_id': 'DEV-2002', 'summary': '원격진단 결과: 배수 문제 가능성 높음 (필터/호스 점검 필요)'}
[AIMessage] 도구 호출 → SupportResponse({'reply': '안녕하세요! 원격진단을 완료하고 결과를 확인했습니다. 진단 결과 배수 문제 가능성이 높으며, 필터와 호스를 점검하는 것이 필요하다고 나타났습니다. 이 결과를 바탕으로 서비스 티켓(TCK-1000)을 생성하여 관련 전문가가 점검을 진행하도록 안내했습니다. 추가적으로 도움이 필요하시면 언제든지 말씀해 주세요!', 'needs_human': False, 'action_items': [{'action': 'ran_diagnosis', 'detail': 'DEV-2002 원격진단 실행'}, {'action': 'created_ticket', 'detail': '원격진단 결과 기반 티켓 생성 (TCK-1000)'}]})
[ToolMessage] Returning structured response: reply='안녕하세요! 원격진단을 완료하고 결과를 확인했습니다. 진단 결과 배수 문제 가능성이 높으며, 필터와 호스를 점검하는 것이 필요하다고 나타났습니다. 이 결

## 📝 실습 과제 (난이도 업)

### 문제 1) 감정(Anger) 감지 → 사람 이관
- 사용자 메시지에 아래 키워드가 포함되면 `needs_human=True`로 설정되도록 **System Prompt**를 개선해보세요.
  - 예: "최악", "법적", "소비자원", "사기", "당장 기사 불러", "환불해" 등

### 문제 2) 기기번호 기억하기 (Multi-turn)
- 1턴에서 기기번호(DEV-xxxx)를 알려주고,
- 2턴에서 "그럼 티켓 만들어줘"라고만 말해도,
- 에이전트가 **이전 기기번호를 기억**하고 처리하도록 테스트 대화를 구성하세요.

### 문제 3) Tool 에러 복구
- 현재 도구들은 권한이 없으면 예외 대신 `AUTH_ERROR: ...` **문자열**을 돌려줍니다. 만약 도구가 `AuthorizationError` **예외**를 던지도록 바꾼다면(실서비스에서 흔한 방식) 에이전트 실행 자체가 중단됩니다.
- 두 경우 모두 에이전트가 **부드럽게 안내**하고 needs_human을 올리도록 프롬프트/설계를 개선하세요. (참고 답안의 방법 A~C)

> 아래 셀은 학생용 TODO입니다.  
> (강사용 답안은 바로 아래에 예시로 제공합니다.)


In [10]:
# ✅ TODO 셀 (학생용)
# - SYSTEM_PROMPT를 개선해보세요.
# - thread_id를 동일하게 유지하여 multi-turn 테스트를 해보세요.

raise NotImplementedError("학생용 TODO 셀입니다. 아래 '참고 답안' 보기 전까지 직접 개선해보세요.")


NotImplementedError: 학생용 TODO 셀입니다. 아래 '참고 답안' 보기 전까지 직접 개선해보세요.

In [ ]:
# ✅ 참고 답안 (핵심만) — 막힐 때만 참고하세요

# --- 문제 1 & 2) 프롬프트 개선(감정 감지/기기ID 기억) + 멀티턴 ---
SYSTEM_PROMPT_V2 = SYSTEM_PROMPT + """
추가 규칙:
- 사용자가 강한 불만/위협/법적 언급("최악","법적","소비자원","환불" 등)을 하면 needs_human=true 로 설정.
- 대화 중 언급된 기기 ID(DEV-xxxx)를 기억하고 유지(다음 턴에 생략돼도 사용).
- 위험 작업은 항상 확인 질문 선행(confirmed=true 필수).
"""

agent_v2 = create_agent(
    model=model,
    tools=[get_device_info, lookup_policy, run_remote_diagnosis, create_service_ticket, push_firmware_update],
    system_prompt=SYSTEM_PROMPT_V2,
    context_schema=SupportContext,
    response_format=ToolStrategy(SupportResponse),
    checkpointer=checkpointer,
)

def invoke_v2(user_text: str, *, thread_id: str, ctx: SupportContext):
    return agent_v2.invoke(
        {"messages": [{"role": "user", "content": user_text}]},
        config={"configurable": {"thread_id": thread_id}, "recursion_limit": 25},
        context=ctx,
    )

# 멀티턴 핵심: thread_id 를 동일하게 유지하면 이전 턴의 기기 ID 가 기억됩니다.
# ctx = SupportContext(user_id="agent-9", user_name="현장지원", role="support_agent", locale="ko")
# print(invoke_v2("DEV-1001 로봇청소기가 자꾸 충돌해요. 원격진단 해줘.", thread_id="mt-1", ctx=ctx)["structured_response"])
# print(invoke_v2("그 기기 티켓도 만들어줘.", thread_id="mt-1", ctx=ctx)["structured_response"])


# --- 문제 3) AuthorizationError 우아한 복구 (핵심 아이디어) ---
# 방법 A) invoke 를 try-except 로 감싸 코드가 직접 안전 응답 생성  (아래 예시)
# 방법 B) wrap_tool_call 미들웨어로 '툴 실행' 단계에서 가로채 END 로 점프
# 방법 C) (09번 실습) LangGraph 노드/라우팅으로 처리
def invoke_agent_safe(user_text: str, *, thread_id: str, ctx: SupportContext):
    try:
        return agent.invoke(
            {"messages": [{"role": "user", "content": user_text}]},
            config={"configurable": {"thread_id": thread_id}, "recursion_limit": 25},
            context=ctx,
        )
    except AuthorizationError as e:
        # LLM 을 거치지 않고 코드가 직접 안전 응답 생성
        return {"structured_response": SupportResponse(
            reply="해당 기능은 전문 상담사만 처리할 수 있습니다. 사람 상담사에게 연결해드리겠습니다.",
            needs_human=True,
            action_items=[ActionItem(action="none", detail=f"AuthorizationError: {e}")],
        )}

## ✅ 정리 & 스스로 점검

**정리**
- 에이전트는 **LLM이 다음 행동(도구 호출 or 최종 답)을 스스로 결정**하는 반복 구조입니다.
- 도구의 이름·설명·타입이 곧 LLM의 설명서입니다. 권한·위험 작업은 **코드로** 막습니다.
- `thread_id`로 대화 메모리를, `recursion_limit`으로 폭주를 관리합니다.

**작은 로컬 모델로 에이전트를 안정화하는 팁**
- 도구 수를 적게, 이름과 docstring을 구체적으로 작성
- 시스템 프롬프트에 "언제 어떤 도구를 쓰는지"를 규칙으로 명시
- `temperature=0`, 출력은 스키마로 강제
- 결과가 이상하면 `print_trace`로 어디서 틀렸는지 확인

**스스로 점검**
1. customer와 support_agent의 실행 기록은 어떻게 달랐나요?
2. `thread_id`를 매번 다르게 주면 "아까 그 세탁기"를 이해할 수 있을까요?
3. 펌웨어 업데이트처럼 위험한 작업에 `confirmed` 인자를 둔 이유는?

**다음 단계 →** 07번: 모델이 모르는 **매뉴얼 지식**을 검색해서 답하게 만듭니다 (RAG).